# Classical Model Comparison

Compare two IMDB sentiment paths on the same cleaned data and stratified validation split:

1. TF-IDF features with logistic regression.
2. Averaged Word2Vec features with logistic regression.

This notebook intentionally omits neural sentiment models.

In [ ]:
# standard library
from pathlib import Path
import sys
from time import perf_counter

# third-party
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# local
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

scripts_path = str(PROJECT_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from imdb_dataset_helpers import clean_reviews, load_datasets
from imdb_model_helpers import (
    average_word_vectors,
    evaluate_classifier,
    split_labeled_reviews,
)

## Configuration

Both paths use these settings and the exact same split. Word2Vec uses one worker to improve reproducibility.

In [ ]:
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
MAX_TFIDF_FEATURES = 100_000
WORD2VEC_DIMENSIONS = 100

## Load, clean, and split

The notebook reloads fresh data and applies the cleanup policy established in Notebook 01. The Kaggle test set is not used during this validation comparison.

In [ ]:
train, _ = load_datasets()
train_clean = clean_reviews(train)

X_train_text, X_validation_text, y_train, y_validation = split_labeled_reviews(
    train_clean,
    validation_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)

pd.DataFrame(
    {
        "rows": {"training": len(X_train_text), "validation": len(X_validation_text)},
        "positive_rate": {
            "training": y_train.mean(),
            "validation": y_validation.mean(),
        },
    }
)

## Path 1: TF-IDF + logistic regression

Fit the vocabulary and inverse-document-frequency weights on the training fold only, then transform validation text with that fitted vectorizer.

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=MAX_TFIDF_FEATURES,
    sublinear_tf=True,
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_validation_tfidf = tfidf.transform(X_validation_text)
X_train_tfidf.shape, X_validation_tfidf.shape

In [ ]:
tfidf_classifier = LogisticRegression(
    max_iter=1_000,
    random_state=RANDOM_STATE,
)

start = perf_counter()
tfidf_classifier.fit(X_train_tfidf, y_train)
tfidf_fit_seconds = perf_counter() - start

tfidf_result = evaluate_classifier(
    name="TF-IDF + logistic regression",
    classifier=tfidf_classifier,
    features=X_validation_tfidf,
    labels=y_validation,
    fit_seconds=tfidf_fit_seconds,
    feature_count=X_train_tfidf.shape[1],
)
pd.Series(tfidf_result)

## Path 2: Word2Vec + logistic regression

Train skip-gram Word2Vec embeddings only on the training fold. Each review becomes the mean of its in-vocabulary word vectors before logistic regression.

In [ ]:
X_train_tokens = [simple_preprocess(review) for review in X_train_text]
X_validation_tokens = [simple_preprocess(review) for review in X_validation_text]

len(X_train_tokens), X_train_tokens[0][:20]

In [ ]:
start = perf_counter()
word2vec = Word2Vec(
    sentences=X_train_tokens,
    vector_size=WORD2VEC_DIMENSIONS,
    window=5,
    min_count=2,
    workers=1,
    sg=1,
    epochs=5,
    seed=RANDOM_STATE,
)

X_train_word2vec = average_word_vectors(X_train_tokens, word2vec.wv)
X_validation_word2vec = average_word_vectors(X_validation_tokens, word2vec.wv)

word2vec_classifier = LogisticRegression(
    max_iter=1_000,
    random_state=RANDOM_STATE,
)
word2vec_classifier.fit(X_train_word2vec, y_train)
word2vec_fit_seconds = perf_counter() - start

word2vec_result = evaluate_classifier(
    name="Word2Vec mean + logistic regression",
    classifier=word2vec_classifier,
    features=X_validation_word2vec,
    labels=y_validation,
    fit_seconds=word2vec_fit_seconds,
    feature_count=X_train_word2vec.shape[1],
)
pd.Series(word2vec_result)

## Side-by-side results

ROC-AUC is the primary comparison. Accuracy and F1 provide complementary threshold-based views; fit time includes embedding training for the Word2Vec path.

In [ ]:
results = (
    pd.DataFrame([tfidf_result, word2vec_result])
    .set_index("model")
    .sort_values("roc_auc", ascending=False)
)
results.round({"roc_auc": 4, "accuracy": 4, "f1": 4, "fit_seconds": 2})

## Interpretation prompts

- Which representation produces the stronger validation ROC-AUC?
- How much performance does each feature dimension buy?
- Is Word2Vec's semantic compression worth its embedding-training time?
- Which model's errors should we inspect before tuning?

Potential follow-ups include TF-IDF hyperparameter tuning, TF-IDF-weighted Word2Vec averages, and use of the unlabeled review corpus for embedding training.